# Data Slayer 4.0 Fine-tuning Notebook

Objective: Fine-tune blok teratas DINOv3 ViT-H+ pada SEMUA frame video train (100% data, tanpa fold hold-out, 15 frame per detik, format test) dengan augmentasi (hide, weather, hflip) untuk mengenali 8 aksi, lalu menyimpan weights untuk inferensi lokal. Weights akhir dirata-rata (EMA) dari kira-kira 20% step terakhir, dan weights tanpa rata-rata juga disimpan.


# Sinkronisasi versi modules dan libraries

In [ ]:
import sys; print(sys.version)
%pip install -q timm==1.0.29

# Import modules dan libraries

In [ ]:
import gc, io, json, math, os, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import cv2, numpy as np, pandas as pd, timm, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from sklearn.metrics import f1_score, roc_auc_score
try:
    from kaggle_secrets import UserSecretsClient
except ImportError:
    UserSecretsClient = None
cv2.setNumThreads(1)
T0 = time.time()
print("timm", timm.__version__, "| torch", torch.__version__)

# Import datasets

### Konfigurasi run
Semua yang boleh diubah ada di sini. Tidak ada fold hold-out: seluruh video train dipakai untuk training dan modelnya dipakai apa adanya (tidak dinilai di sini, dibandingkan dengan 1.csv di lokal).
TIME_BUDGET_H adalah batas total sesi: jumlah step dipotong otomatis agar training selesai di dalam batas ini.


In [ ]:
K = 12                  # jumlah blok ViT-H+ teratas (dari 32) yang dilatih
FPS = 15                # frame per detik video yang dipakai untuk training (30 = semua frame, 15 = tiap frame ke-2)
EPOCHS = 2
TIME_BUDGET_H = 8.5     # batas total sesi dalam jam (turunkan ke 8 kalau sesi GPU 9 jam, naikkan ke 11 kalau 12 jam)
EMA_FRAC = 0.2          # weights akhir = rata-rata (EMA) dari kira-kira EMA_FRAC terakhir step training
AUG = "hide+weather+hflip"   # augmentasi training: hide = erase/cutout/griddrop/coarse, weather = fog/shadow/flare/rain/snow, hflip
BS, LR_BLOCKS, LR_HEAD, WD, SEED = 16, 2e-5, 1e-3, 0.05, 42
IMG_HW, DEVICE = (224, 400), "cuda"
OUT = Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
STEP = 30 // FPS
ACTIONS = ["Looking_Under_the_Desk", "Reading_Hidden_Notes", "Repeated_Peeking", "Using_Smartwatch", "Checking_Smartwatch", "Looking_Around", "Thinking", "Writing_Answers"]
WRITE = ACTIONS.index("Writing_Answers")
torch.manual_seed(SEED); np.random.seed(SEED)
print("100% train |", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB", "| capability", torch.cuda.get_device_capability(0))


### Tabel video dan waktu mulai aksi
Start adalah detik mulai aksi hasil pengecekan mata (kolom fold tidak dipakai di notebook ini). Tabel ini ditempel langsung supaya tidak bergantung pada urutan file di Kaggle.


In [ ]:
FOLD_TABLE_CSV = """video,fold,action_id,start,label
cheat/subject_1/front_Looking_Under_the_Desk,2,0,6.75,1
cheat/subject_1/front_Reading_Hidden_Notes,2,1,5.25,1
cheat/subject_1/front_Repeated_Peeking,2,2,6.25,1
cheat/subject_1/front_Using_Smartwatch,2,3,6.75,1
cheat/subject_1/side_Looking_Under_the_Desk,2,0,7.25,1
cheat/subject_1/side_Reading_Hidden_Notes,2,1,5.75,1
cheat/subject_1/side_Repeated_Peeking,2,2,6.75,1
cheat/subject_1/side_Using_Smartwatch,2,3,7.25,1
cheat/subject_10/front_Looking_Under_the_Desk,0,0,,1
cheat/subject_10/front_Reading_Hidden_Notes,0,1,7.75,1
cheat/subject_10/front_Repeated_Peeking,0,2,7.75,1
cheat/subject_10/front_Using_Smartwatch,0,3,7.75,1
cheat/subject_10/side_Looking_Under_the_Desk,0,0,,1
cheat/subject_10/side_Reading_Hidden_Notes,0,1,8.25,1
cheat/subject_10/side_Repeated_Peeking,0,2,8.25,1
cheat/subject_10/side_Using_Smartwatch,0,3,8.25,1
cheat/subject_11/front_Looking_Under_the_Desk,2,0,,1
cheat/subject_11/front_Reading_Hidden_Notes,2,1,7.25,1
cheat/subject_11/front_Repeated_Peeking,2,2,6.75,1
cheat/subject_11/front_Using_Smartwatch,2,3,7.75,1
cheat/subject_11/side_Looking_Under_the_Desk,2,0,,1
cheat/subject_11/side_Reading_Hidden_Notes,2,1,7.75,1
cheat/subject_11/side_Repeated_Peeking,2,2,7.25,1
cheat/subject_11/side_Using_Smartwatch,2,3,8.25,1
cheat/subject_12/front_Looking_Under_the_Desk,4,0,8.25,1
cheat/subject_12/front_Reading_Hidden_Notes,4,1,8.75,1
cheat/subject_12/front_Repeated_Peeking,4,2,7.25,1
cheat/subject_12/front_Using_Smartwatch,4,3,7.75,1
cheat/subject_12/side_Looking_Under_the_Desk,4,0,8.75,1
cheat/subject_12/side_Reading_Hidden_Notes,4,1,9.25,1
cheat/subject_12/side_Repeated_Peeking,4,2,7.75,1
cheat/subject_12/side_Using_Smartwatch,4,3,8.25,1
cheat/subject_13/front_Looking_Under_the_Desk,1,0,,1
cheat/subject_13/front_Reading_Hidden_Notes,1,1,8.75,1
cheat/subject_13/front_Repeated_Peeking,1,2,7.75,1
cheat/subject_13/front_Using_Smartwatch,1,3,8.25,1
cheat/subject_13/side_Looking_Under_the_Desk,1,0,,1
cheat/subject_13/side_Reading_Hidden_Notes,1,1,9.25,1
cheat/subject_13/side_Repeated_Peeking,1,2,8.25,1
cheat/subject_13/side_Using_Smartwatch,1,3,8.75,1
cheat/subject_14/front_Looking_Under_the_Desk,1,0,7.75,1
cheat/subject_14/front_Reading_Hidden_Notes,1,1,6.75,1
cheat/subject_14/front_Repeated_Peeking,1,2,7.25,1
cheat/subject_14/front_Using_Smartwatch,1,3,7.25,1
cheat/subject_14/side_Looking_Under_the_Desk,1,0,8.25,1
cheat/subject_14/side_Reading_Hidden_Notes,1,1,7.25,1
cheat/subject_14/side_Repeated_Peeking,1,2,7.75,1
cheat/subject_14/side_Using_Smartwatch,1,3,7.75,1
cheat/subject_15/front_Looking_Under_the_Desk,3,0,7.75,1
cheat/subject_15/front_Reading_Hidden_Notes,3,1,7.25,1
cheat/subject_15/front_Repeated_Peeking,3,2,7.25,1
cheat/subject_15/front_Using_Smartwatch,3,3,7.75,1
cheat/subject_15/side_Looking_Under_the_Desk,3,0,8.25,1
cheat/subject_15/side_Reading_Hidden_Notes,3,1,7.75,1
cheat/subject_15/side_Repeated_Peeking,3,2,7.75,1
cheat/subject_15/side_Using_Smartwatch,3,3,8.25,1
cheat/subject_16/front_Looking_Under_the_Desk,2,0,7.25,1
cheat/subject_16/front_Reading_Hidden_Notes,2,1,6.75,1
cheat/subject_16/front_Repeated_Peeking,2,2,7.25,1
cheat/subject_16/front_Using_Smartwatch,2,3,7.75,1
cheat/subject_16/side_Looking_Under_the_Desk,2,0,7.75,1
cheat/subject_16/side_Reading_Hidden_Notes,2,1,7.25,1
cheat/subject_16/side_Repeated_Peeking,2,2,7.75,1
cheat/subject_16/side_Using_Smartwatch,2,3,8.25,1
cheat/subject_17/front_Looking_Under_the_Desk,1,0,7.75,1
cheat/subject_17/front_Reading_Hidden_Notes,1,1,7.75,1
cheat/subject_17/front_Repeated_Peeking,1,2,8.25,1
cheat/subject_17/front_Using_Smartwatch,1,3,7.75,1
cheat/subject_17/side_Looking_Under_the_Desk,1,0,8.25,1
cheat/subject_17/side_Reading_Hidden_Notes,1,1,8.25,1
cheat/subject_17/side_Repeated_Peeking,1,2,8.75,1
cheat/subject_17/side_Using_Smartwatch,1,3,8.25,1
cheat/subject_18/front_Looking_Under_the_Desk,1,0,7.75,1
cheat/subject_18/front_Reading_Hidden_Notes,1,1,8.25,1
cheat/subject_18/front_Repeated_Peeking,1,2,7.25,1
cheat/subject_18/front_Using_Smartwatch,1,3,7.25,1
cheat/subject_18/side_Looking_Under_the_Desk,1,0,8.25,1
cheat/subject_18/side_Reading_Hidden_Notes,1,1,8.75,1
cheat/subject_18/side_Repeated_Peeking,1,2,7.75,1
cheat/subject_18/side_Using_Smartwatch,1,3,7.75,1
cheat/subject_19/front_Looking_Under_the_Desk,3,0,7.75,1
cheat/subject_19/front_Reading_Hidden_Notes,3,1,7.25,1
cheat/subject_19/front_Repeated_Peeking,3,2,7.25,1
cheat/subject_19/front_Using_Smartwatch,3,3,7.75,1
cheat/subject_19/side_Looking_Under_the_Desk,3,0,8.25,1
cheat/subject_19/side_Reading_Hidden_Notes,3,1,7.75,1
cheat/subject_19/side_Repeated_Peeking,3,2,7.75,1
cheat/subject_19/side_Using_Smartwatch,3,3,8.25,1
cheat/subject_2/front_Looking_Under_the_Desk,2,0,7.75,1
cheat/subject_2/front_Reading_Hidden_Notes,2,1,7.25,1
cheat/subject_2/front_Repeated_Peeking,2,2,7.25,1
cheat/subject_2/front_Using_Smartwatch,2,3,7.25,1
cheat/subject_2/side_Looking_Under_the_Desk,2,0,8.25,1
cheat/subject_2/side_Reading_Hidden_Notes,2,1,7.75,1
cheat/subject_2/side_Repeated_Peeking,2,2,7.75,1
cheat/subject_2/side_Using_Smartwatch,2,3,7.75,1
cheat/subject_20/front_Looking_Under_the_Desk,4,0,,1
cheat/subject_20/front_Reading_Hidden_Notes,4,1,8.25,1
cheat/subject_20/front_Repeated_Peeking,4,2,7.75,1
cheat/subject_20/front_Using_Smartwatch,4,3,7.75,1
cheat/subject_20/side_Looking_Under_the_Desk,4,0,,1
cheat/subject_20/side_Reading_Hidden_Notes,4,1,8.75,1
cheat/subject_20/side_Repeated_Peeking,4,2,8.25,1
cheat/subject_20/side_Using_Smartwatch,4,3,8.25,1
cheat/subject_21/front_Looking_Under_the_Desk,0,0,,1
cheat/subject_21/front_Reading_Hidden_Notes,0,1,7.75,1
cheat/subject_21/front_Repeated_Peeking,0,2,7.75,1
cheat/subject_21/front_Using_Smartwatch,0,3,7.75,1
cheat/subject_21/side_Looking_Under_the_Desk,0,0,,1
cheat/subject_21/side_Reading_Hidden_Notes,0,1,8.25,1
cheat/subject_21/side_Repeated_Peeking,0,2,8.25,1
cheat/subject_21/side_Using_Smartwatch,0,3,8.25,1
cheat/subject_22/front_Looking_Under_the_Desk,0,0,6.25,1
cheat/subject_22/front_Reading_Hidden_Notes,0,1,6.25,1
cheat/subject_22/front_Repeated_Peeking,0,2,6.25,1
cheat/subject_22/front_Using_Smartwatch,0,3,6.25,1
cheat/subject_22/side_Looking_Under_the_Desk,0,0,6.75,1
cheat/subject_22/side_Reading_Hidden_Notes,0,1,6.75,1
cheat/subject_22/side_Repeated_Peeking,0,2,6.75,1
cheat/subject_22/side_Using_Smartwatch,0,3,6.75,1
cheat/subject_23/front_Looking_Under_the_Desk,2,0,7.25,1
cheat/subject_23/front_Reading_Hidden_Notes,2,1,7.75,1
cheat/subject_23/front_Repeated_Peeking,2,2,8.25,1
cheat/subject_23/front_Using_Smartwatch,2,3,8.25,1
cheat/subject_23/side_Looking_Under_the_Desk,2,0,7.75,1
cheat/subject_23/side_Reading_Hidden_Notes,2,1,8.25,1
cheat/subject_23/side_Repeated_Peeking,2,2,8.75,1
cheat/subject_23/side_Using_Smartwatch,2,3,8.75,1
cheat/subject_24/front_Looking_Under_the_Desk,3,0,,1
cheat/subject_24/front_Reading_Hidden_Notes,3,1,7.75,1
cheat/subject_24/front_Repeated_Peeking,3,2,7.25,1
cheat/subject_24/front_Using_Smartwatch,3,3,7.75,1
cheat/subject_24/side_Looking_Under_the_Desk,3,0,,1
cheat/subject_24/side_Reading_Hidden_Notes,3,1,8.25,1
cheat/subject_24/side_Repeated_Peeking,3,2,7.75,1
cheat/subject_24/side_Using_Smartwatch,3,3,8.25,1
cheat/subject_25/front_Looking_Under_the_Desk,3,0,7.75,1
cheat/subject_25/front_Reading_Hidden_Notes,3,1,7.25,1
cheat/subject_25/front_Repeated_Peeking,3,2,7.75,1
cheat/subject_25/front_Using_Smartwatch,3,3,8.25,1
cheat/subject_25/side_Looking_Under_the_Desk,3,0,8.25,1
cheat/subject_25/side_Reading_Hidden_Notes,3,1,7.75,1
cheat/subject_25/side_Repeated_Peeking,3,2,8.25,1
cheat/subject_25/side_Using_Smartwatch,3,3,8.75,1
cheat/subject_26/front_Looking_Under_the_Desk,4,0,,1
cheat/subject_26/front_Reading_Hidden_Notes,4,1,8.25,1
cheat/subject_26/front_Repeated_Peeking,4,2,8.25,1
cheat/subject_26/front_Using_Smartwatch,4,3,8.25,1
cheat/subject_26/side_Looking_Under_the_Desk,4,0,,1
cheat/subject_26/side_Reading_Hidden_Notes,4,1,8.75,1
cheat/subject_26/side_Repeated_Peeking,4,2,8.75,1
cheat/subject_26/side_Using_Smartwatch,4,3,8.75,1
cheat/subject_3/front_Looking_Under_the_Desk,0,0,6.75,1
cheat/subject_3/front_Reading_Hidden_Notes,0,1,6.75,1
cheat/subject_3/front_Repeated_Peeking,0,2,5.75,1
cheat/subject_3/front_Using_Smartwatch,0,3,6.75,1
cheat/subject_3/side_Looking_Under_the_Desk,0,0,7.25,1
cheat/subject_3/side_Reading_Hidden_Notes,0,1,7.25,1
cheat/subject_3/side_Repeated_Peeking,0,2,6.25,1
cheat/subject_3/side_Using_Smartwatch,0,3,7.25,1
cheat/subject_4/front_Looking_Under_the_Desk,3,0,6.75,1
cheat/subject_4/front_Reading_Hidden_Notes,3,1,6.25,1
cheat/subject_4/front_Repeated_Peeking,3,2,4.75,1
cheat/subject_4/front_Using_Smartwatch,3,3,6.25,1
cheat/subject_4/side_Looking_Under_the_Desk,3,0,7.25,1
cheat/subject_4/side_Reading_Hidden_Notes,3,1,6.75,1
cheat/subject_4/side_Repeated_Peeking,3,2,5.25,1
cheat/subject_4/side_Using_Smartwatch,3,3,6.75,1
cheat/subject_5/front_Looking_Under_the_Desk,4,0,7.25,1
cheat/subject_5/front_Reading_Hidden_Notes,4,1,7.25,1
cheat/subject_5/front_Repeated_Peeking,4,2,6.75,1
cheat/subject_5/front_Using_Smartwatch,4,3,7.25,1
cheat/subject_5/side_Looking_Under_the_Desk,4,0,7.75,1
cheat/subject_5/side_Reading_Hidden_Notes,4,1,7.75,1
cheat/subject_5/side_Repeated_Peeking,4,2,7.25,1
cheat/subject_5/side_Using_Smartwatch,4,3,7.75,1
cheat/subject_6/front_Looking_Under_the_Desk,4,0,7.75,1
cheat/subject_6/front_Reading_Hidden_Notes,4,1,7.25,1
cheat/subject_6/front_Repeated_Peeking,4,2,7.25,1
cheat/subject_6/front_Using_Smartwatch,4,3,7.25,1
cheat/subject_6/side_Looking_Under_the_Desk,4,0,8.25,1
cheat/subject_6/side_Reading_Hidden_Notes,4,1,7.75,1
cheat/subject_6/side_Repeated_Peeking,4,2,7.75,1
cheat/subject_6/side_Using_Smartwatch,4,3,7.75,1
cheat/subject_7/front_Looking_Under_the_Desk,0,0,7.25,1
cheat/subject_7/front_Reading_Hidden_Notes,0,1,7.75,1
cheat/subject_7/front_Repeated_Peeking,0,2,7.25,1
cheat/subject_7/front_Using_Smartwatch,0,3,7.75,1
cheat/subject_7/side_Looking_Under_the_Desk,0,0,7.75,1
cheat/subject_7/side_Reading_Hidden_Notes,0,1,8.25,1
cheat/subject_7/side_Repeated_Peeking,0,2,7.75,1
cheat/subject_7/side_Using_Smartwatch,0,3,8.25,1
cheat/subject_8/front_Looking_Under_the_Desk,1,0,7.75,1
cheat/subject_8/front_Reading_Hidden_Notes,1,1,7.75,1
cheat/subject_8/front_Repeated_Peeking,1,2,5.75,1
cheat/subject_8/front_Using_Smartwatch,1,3,7.75,1
cheat/subject_8/side_Looking_Under_the_Desk,1,0,8.25,1
cheat/subject_8/side_Reading_Hidden_Notes,1,1,8.25,1
cheat/subject_8/side_Repeated_Peeking,1,2,6.25,1
cheat/subject_8/side_Using_Smartwatch,1,3,8.25,1
cheat/subject_9/front_Looking_Under_the_Desk,0,0,,1
cheat/subject_9/front_Reading_Hidden_Notes,0,1,,1
cheat/subject_9/front_Repeated_Peeking,0,2,7.75,1
cheat/subject_9/front_Using_Smartwatch,0,3,7.75,1
cheat/subject_9/side_Looking_Under_the_Desk,0,0,,1
cheat/subject_9/side_Reading_Hidden_Notes,0,1,,1
cheat/subject_9/side_Repeated_Peeking,0,2,8.25,1
cheat/subject_9/side_Using_Smartwatch,0,3,8.25,1
normal/subject_1/front_Checking_Smartwatch,2,4,,0
normal/subject_1/front_Looking_Around,2,5,,0
normal/subject_1/front_Thinking,2,6,,0
normal/subject_1/front_Writing_Answers,2,7,,0
normal/subject_1/side_Checking_Smartwatch,2,4,,0
normal/subject_1/side_Looking_Around,2,5,,0
normal/subject_1/side_Thinking,2,6,,0
normal/subject_1/side_Writing_Answers,2,7,,0
normal/subject_10/front_Checking_Smartwatch,0,4,,0
normal/subject_10/front_Looking_Around,0,5,,0
normal/subject_10/front_Thinking,0,6,,0
normal/subject_10/front_Writing_Answers,0,7,,0
normal/subject_10/side_Checking_Smartwatch,0,4,,0
normal/subject_10/side_Looking_Around,0,5,,0
normal/subject_10/side_Thinking,0,6,,0
normal/subject_10/side_Writing_Answers,0,7,,0
normal/subject_11/front_Checking_Smartwatch,2,4,,0
normal/subject_11/front_Looking_Around,2,5,,0
normal/subject_11/front_Thinking,2,6,,0
normal/subject_11/front_Writing_Answers,2,7,,0
normal/subject_11/side_Checking_Smartwatch,2,4,,0
normal/subject_11/side_Looking_Around,2,5,,0
normal/subject_11/side_Thinking,2,6,,0
normal/subject_11/side_Writing_Answers,2,7,,0
normal/subject_12/front_Checking_Smartwatch,4,4,,0
normal/subject_12/front_Looking_Around,4,5,,0
normal/subject_12/front_Thinking,4,6,,0
normal/subject_12/front_Writing_Answers,4,7,,0
normal/subject_12/side_Checking_Smartwatch,4,4,,0
normal/subject_12/side_Looking_Around,4,5,,0
normal/subject_12/side_Thinking,4,6,,0
normal/subject_12/side_Writing_Answers,4,7,,0
normal/subject_13/front_Checking_Smartwatch,1,4,,0
normal/subject_13/front_Looking_Around,1,5,,0
normal/subject_13/front_Thinking,1,6,,0
normal/subject_13/front_Writing_Answers,1,7,,0
normal/subject_13/side_Checking_Smartwatch,1,4,,0
normal/subject_13/side_Looking_Around,1,5,,0
normal/subject_13/side_Thinking,1,6,,0
normal/subject_13/side_Writing_Answers,1,7,,0
normal/subject_14/front_Checking_Smartwatch,1,4,,0
normal/subject_14/front_Looking_Around,1,5,,0
normal/subject_14/front_Thinking,1,6,,0
normal/subject_14/front_Writing_Answers,1,7,,0
normal/subject_14/side_Checking_Smartwatch,1,4,,0
normal/subject_14/side_Looking_Around,1,5,,0
normal/subject_14/side_Thinking,1,6,,0
normal/subject_14/side_Writing_Answers,1,7,,0
normal/subject_15/front_Checking_Smartwatch,3,4,,0
normal/subject_15/front_Looking_Around,3,5,,0
normal/subject_15/front_Thinking,3,6,,0
normal/subject_15/front_Writing_Answers,3,7,,0
normal/subject_15/side_Checking_Smartwatch,3,4,,0
normal/subject_15/side_Looking_Around,3,5,,0
normal/subject_15/side_Thinking,3,6,,0
normal/subject_15/side_Writing_Answers,3,7,,0
normal/subject_16/front_Checking_Smartwatch,2,4,,0
normal/subject_16/front_Looking_Around,2,5,,0
normal/subject_16/front_Thinking,2,6,,0
normal/subject_16/front_Writing_Answers,2,7,,0
normal/subject_16/side_Checking_Smartwatch,2,4,,0
normal/subject_16/side_Looking_Around,2,5,,0
normal/subject_16/side_Thinking,2,6,,0
normal/subject_16/side_Writing_Answers,2,7,,0
normal/subject_17/front_Checking_Smartwatch,1,4,,0
normal/subject_17/front_Looking_Around,1,5,,0
normal/subject_17/front_Thinking,1,6,,0
normal/subject_17/front_Writing_Answers,1,7,,0
normal/subject_17/side_Checking_Smartwatch,1,4,,0
normal/subject_17/side_Looking_Around,1,5,,0
normal/subject_17/side_Thinking,1,6,,0
normal/subject_17/side_Writing_Answers,1,7,,0
normal/subject_18/front_Checking_Smartwatch,1,4,,0
normal/subject_18/front_Looking_Around,1,5,,0
normal/subject_18/front_Thinking,1,6,,0
normal/subject_18/front_Writing_Answers,1,7,,0
normal/subject_18/side_Checking_Smartwatch,1,4,,0
normal/subject_18/side_Looking_Around,1,5,,0
normal/subject_18/side_Thinking,1,6,,0
normal/subject_18/side_Writing_Answers,1,7,,0
normal/subject_19/front_Checking_Smartwatch,3,4,,0
normal/subject_19/front_Looking_Around,3,5,,0
normal/subject_19/front_Thinking,3,6,,0
normal/subject_19/front_Writing_Answers,3,7,,0
normal/subject_19/side_Checking_Smartwatch,3,4,,0
normal/subject_19/side_Looking_Around,3,5,,0
normal/subject_19/side_Thinking,3,6,,0
normal/subject_19/side_Writing_Answers,3,7,,0
normal/subject_2/front_Checking_Smartwatch,2,4,,0
normal/subject_2/front_Looking_Around,2,5,,0
normal/subject_2/front_Thinking,2,6,,0
normal/subject_2/front_Writing_Answers,2,7,,0
normal/subject_2/side_Checking_Smartwatch,2,4,,0
normal/subject_2/side_Looking_Around,2,5,,0
normal/subject_2/side_Thinking,2,6,,0
normal/subject_2/side_Writing_Answers,2,7,,0
normal/subject_20/front_Checking_Smartwatch,4,4,,0
normal/subject_20/front_Looking_Around,4,5,,0
normal/subject_20/front_Thinking,4,6,,0
normal/subject_20/front_Writing_Answers,4,7,,0
normal/subject_20/side_Checking_Smartwatch,4,4,,0
normal/subject_20/side_Looking_Around,4,5,,0
normal/subject_20/side_Thinking,4,6,,0
normal/subject_20/side_Writing_Answers,4,7,,0
normal/subject_21/front_Checking_Smartwatch,0,4,,0
normal/subject_21/front_Looking_Around,0,5,,0
normal/subject_21/front_Thinking,0,6,,0
normal/subject_21/front_Writing_Answers,0,7,,0
normal/subject_21/side_Checking_Smartwatch,0,4,,0
normal/subject_21/side_Looking_Around,0,5,,0
normal/subject_21/side_Thinking,0,6,,0
normal/subject_21/side_Writing_Answers,0,7,,0
normal/subject_22/front_Checking_Smartwatch,0,4,,0
normal/subject_22/front_Looking_Around,0,5,,0
normal/subject_22/front_Thinking,0,6,,0
normal/subject_22/front_Writing_Answers,0,7,,0
normal/subject_22/side_Checking_Smartwatch,0,4,,0
normal/subject_22/side_Looking_Around,0,5,,0
normal/subject_22/side_Thinking,0,6,,0
normal/subject_22/side_Writing_Answers,0,7,,0
normal/subject_23/front_Checking_Smartwatch,2,4,,0
normal/subject_23/front_Looking_Around,2,5,,0
normal/subject_23/front_Thinking,2,6,,0
normal/subject_23/front_Writing_Answers,2,7,,0
normal/subject_23/side_Checking_Smartwatch,2,4,,0
normal/subject_23/side_Looking_Around,2,5,,0
normal/subject_23/side_Thinking,2,6,,0
normal/subject_23/side_Writing_Answers,2,7,,0
normal/subject_24/front_Checking_Smartwatch,3,4,,0
normal/subject_24/front_Looking_Around,3,5,,0
normal/subject_24/front_Thinking,3,6,,0
normal/subject_24/front_Writing_Answers,3,7,,0
normal/subject_24/side_Checking_Smartwatch,3,4,,0
normal/subject_24/side_Looking_Around,3,5,,0
normal/subject_24/side_Thinking,3,6,,0
normal/subject_24/side_Writing_Answers,3,7,,0
normal/subject_25/front_Checking_Smartwatch,3,4,,0
normal/subject_25/front_Looking_Around,3,5,,0
normal/subject_25/front_Thinking,3,6,,0
normal/subject_25/front_Writing_Answers,3,7,,0
normal/subject_25/side_Checking_Smartwatch,3,4,,0
normal/subject_25/side_Looking_Around,3,5,,0
normal/subject_25/side_Thinking,3,6,,0
normal/subject_25/side_Writing_Answers,3,7,,0
normal/subject_26/front_Checking_Smartwatch,4,4,,0
normal/subject_26/front_Looking_Around,4,5,,0
normal/subject_26/front_Thinking,4,6,,0
normal/subject_26/front_Writing_Answers,4,7,,0
normal/subject_26/side_Checking_Smartwatch,4,4,,0
normal/subject_26/side_Looking_Around,4,5,,0
normal/subject_26/side_Thinking,4,6,,0
normal/subject_26/side_Writing_Answers,4,7,,0
normal/subject_3/front_Checking_Smartwatch,0,4,,0
normal/subject_3/front_Looking_Around,0,5,,0
normal/subject_3/front_Thinking,0,6,,0
normal/subject_3/front_Writing_Answers,0,7,,0
normal/subject_3/side_Checking_Smartwatch,0,4,,0
normal/subject_3/side_Looking_Around,0,5,,0
normal/subject_3/side_Thinking,0,6,,0
normal/subject_3/side_Writing_Answers,0,7,,0
normal/subject_4/front_Checking_Smartwatch,3,4,,0
normal/subject_4/front_Looking_Around,3,5,,0
normal/subject_4/front_Thinking,3,6,,0
normal/subject_4/front_Writing_Answers,3,7,,0
normal/subject_4/side_Checking_Smartwatch,3,4,,0
normal/subject_4/side_Looking_Around,3,5,,0
normal/subject_4/side_Thinking,3,6,,0
normal/subject_4/side_Writing_Answers,3,7,,0
normal/subject_5/front_Checking_Smartwatch,4,4,,0
normal/subject_5/front_Looking_Around,4,5,,0
normal/subject_5/front_Thinking,4,6,,0
normal/subject_5/front_Writing_Answers,4,7,,0
normal/subject_5/side_Checking_Smartwatch,4,4,,0
normal/subject_5/side_Looking_Around,4,5,,0
normal/subject_5/side_Thinking,4,6,,0
normal/subject_5/side_Writing_Answers,4,7,,0
normal/subject_6/front_Checking_Smartwatch,4,4,,0
normal/subject_6/front_Looking_Around,4,5,,0
normal/subject_6/front_Thinking,4,6,,0
normal/subject_6/front_Writing_Answers,4,7,,0
normal/subject_6/side_Checking_Smartwatch,4,4,,0
normal/subject_6/side_Looking_Around,4,5,,0
normal/subject_6/side_Thinking,4,6,,0
normal/subject_6/side_Writing_Answers,4,7,,0
normal/subject_7/front_Checking_Smartwatch,0,4,,0
normal/subject_7/front_Looking_Around,0,5,,0
normal/subject_7/front_Thinking,0,6,,0
normal/subject_7/front_Writing_Answers,0,7,,0
normal/subject_7/side_Checking_Smartwatch,0,4,,0
normal/subject_7/side_Looking_Around,0,5,,0
normal/subject_7/side_Thinking,0,6,,0
normal/subject_7/side_Writing_Answers,0,7,,0
normal/subject_8/front_Checking_Smartwatch,1,4,,0
normal/subject_8/front_Looking_Around,1,5,,0
normal/subject_8/front_Thinking,1,6,,0
normal/subject_8/front_Writing_Answers,1,7,,0
normal/subject_8/side_Checking_Smartwatch,1,4,,0
normal/subject_8/side_Looking_Around,1,5,,0
normal/subject_8/side_Thinking,1,6,,0
normal/subject_8/side_Writing_Answers,1,7,,0
normal/subject_9/front_Checking_Smartwatch,0,4,,0
normal/subject_9/front_Looking_Around,0,5,,0
normal/subject_9/front_Thinking,0,6,,0
normal/subject_9/front_Writing_Answers,0,7,,0
normal/subject_9/side_Checking_Smartwatch,0,4,,0
normal/subject_9/side_Looking_Around,0,5,,0
normal/subject_9/side_Thinking,0,6,,0
normal/subject_9/side_Writing_Answers,0,7,,0"""
table = pd.read_csv(io.StringIO(FOLD_TABLE_CSV))
found = {}
for p in list(Path("/kaggle/input").rglob("*.MOV")) + list(Path("/kaggle/input").rglob("*.mov")):
    found[f"{p.parent.parent.name}/{p.parent.name}/{p.stem}"] = p
table["path"] = table["video"].map(found)
assert table["path"].notna().all(), table.loc[table["path"].isna(), "video"].tolist()[:5]
print(len(table), "video train ditemukan")

# EDA

In [ ]:
table.groupby("label").agg(video=("video", "count"), cheat=("label", "sum"))

# Preprocessing

### Frame dalam format test
Frame diubah ke 640x360 lalu di-JPEG kualitas 90, sama seperti frame test, dan disimpan sebagai bytes JPEG di RAM.


In [ ]:
def read_video(path):
    cap = cv2.VideoCapture(str(path)); fps = cap.get(cv2.CAP_PROP_FPS) or 30.0; out, i = [], 0
    while cap.grab():
        if i % STEP == 0:
            ok, img = cap.retrieve()
            if ok:
                img = cv2.resize(img, (640, 360), interpolation=cv2.INTER_AREA)
                out.append((i, cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 90])[1].tobytes()))
        i += 1
    cap.release()
    return out, fps


jpegs, rows = [], []
with ThreadPoolExecutor(4) as pool:
    for k, ((frames, fps), r) in enumerate(zip(pool.map(read_video, table["path"]), table.itertuples())):
        for i, b in frames:
            jpegs.append(b); rows.append((r.video, i, i / fps))
        if k % 50 == 0:
            print(f"video {k}/{len(table)} | {len(jpegs)} frame | {(time.time() - T0) / 60:.1f} menit", flush=True)
meta = pd.DataFrame(rows, columns=["video", "frame", "sec"]).merge(table[["video", "action_id", "start"]], on="video", how="left")
print(len(meta), "frame |", f"{sum(map(len, jpegs)) / 1e9:.1f} GB JPEG di RAM")

### Label per frame
Sebelum aksi dimulai orang menulis (Writing_Answers), sesudahnya aksi video itu, dan 0.5 detik di sekitar start dibuang (-1).

In [ ]:
def frame_labels(df):
    lab = np.full(len(df), -1)
    for i, (a, s, t) in enumerate(zip(df.action_id, df.start, df.sec)):
        if a == WRITE:
            lab[i] = WRITE
        elif not np.isnan(s):
            lab[i] = a if t >= s + 0.5 else (WRITE if t < s - 0.5 else -1)
        else:
            lab[i] = a if t >= 9.75 else (WRITE if t < 4.25 else -1)
    return lab


meta["y"] = frame_labels(meta)
print("frame berlabel", int((meta["y"].values >= 0).sum()), "dari", len(meta))

# Augmentation

### Augmentasi gambar per batch
Setiap frame training diubah acak di GPU sebelum masuk model: hide menutup bagian gambar (erase, cutout, griddrop, coarse), weather menambah kabut, bayangan, flare, hujan, salju, dan hflip membalik gambar kiri-kanan. Hasil screen lokal fold 0: AUC 0.8495 tanpa augmentasi menjadi 0.8866.

In [ ]:
GROUPS = {
    "geo": "zoom,shift,rot,shear,persp,elastic",
    "photo": "bright,contrast,sat,hue,gamma,gray,autocontrast,posterize,equalize",
    "quality": "blur,motion,noise,sp,jpeg,downup,sharpen,patch",
    "hide": "erase,cutout,griddrop,coarse",
    "mix": "mixup,cutmix",
    "weather": "fog,shadow,flare,rain,snow",
    "legacy1": "bright,contrast,sat,zoom,shift,rot,blur",
}
ALL = set(",".join(GROUPS.values()).split(",")) | {"crop", "hflip"}


def parse(spec):
    ops = []
    for t in spec.replace("+", ",").split(","):
        t = t.strip()
        if t in ("", "0", "none"):
            continue
        if t in GROUPS:
            ops += GROUPS[t].split(",")
        else:
            assert t in ALL, f"unknown augmentation {t}"
            ops.append(t)
    return list(dict.fromkeys(ops))


def _u(x, lo, hi):
    return lo + (hi - lo) * torch.rand(x.shape[0], 1, 1, 1, device=x.device)


def _p(x, p):
    return (torch.rand(x.shape[0], 1, 1, 1, device=x.device) < p).float()


def _gray(x):
    return (0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3])


def _conv(x, k):
    k = k.to(x.device, x.dtype); r = k.shape[-1] // 2
    return F.conv2d(F.pad(x, (r, r, r, r), mode="reflect"), k.expand(3, 1, *k.shape[-2:]).contiguous(), groups=3)


def _gauss_k(sigma, size=5):
    a = torch.exp(-(torch.arange(size) - size // 2) ** 2 / (2 * sigma ** 2)); a = a / a.sum()
    return (a[:, None] * a[None, :])[None, None]


def _line_k(angle, length, size=9):
    k = torch.zeros(size, size); c = size // 2
    for t in np.linspace(-length / 2, length / 2, 6 * size):
        i, j = int(round(c + t * math.sin(angle))), int(round(c + t * math.cos(angle)))
        if 0 <= i < size and 0 <= j < size:
            k[i, j] = 1
    return (k / k.sum())[None, None]


def _blob(x, sig_lo, sig_hi):
    B, _, H, W = x.shape
    X = torch.linspace(0, 1, W, device=x.device).view(1, 1, 1, W); Y = (torch.linspace(0, 1, H, device=x.device) * H / W).view(1, 1, H, 1)
    cx, cy = _u(x, 0, 1), _u(x, 0, H / W)
    s = _u(x, sig_lo, sig_hi)
    return torch.exp(-((X - cx) ** 2 + (Y - cy) ** 2) / (2 * s ** 2))


def _boxes(x, n, hlo, hhi, wlo, whi):
    B, _, H, W = x.shape
    m = torch.zeros(B, 1, H, W, dtype=torch.bool, device=x.device)
    Y = torch.arange(H, device=x.device).view(1, 1, H, 1); X = torch.arange(W, device=x.device).view(1, 1, 1, W)
    for _ in range(n):
        h, w = _u(x, hlo, hhi) * H, _u(x, wlo, whi) * W
        y0, x0 = torch.rand(B, 1, 1, 1, device=x.device) * (H - h), torch.rand(B, 1, 1, 1, device=x.device) * (W - w)
        m |= (Y >= y0) & (Y < y0 + h) & (X >= x0) & (X < x0 + w)
    return m


def geometry(x, ops):
    if not any(o in ops for o in ("zoom", "crop", "shift", "rot", "shear", "persp", "elastic", "hflip")):
        return x
    B, _, H, W = x.shape; dev = x.device
    if "hflip" in ops:
        x = torch.where(torch.rand(B, 1, 1, 1, device=dev) < 0.5, x.flip(3), x)
    r = lambda lo, hi: lo + (hi - lo) * torch.rand(B, device=dev)
    sx = sy = torch.ones(B, device=dev); tx = ty = torch.zeros(B, device=dev)
    if "zoom" in ops:
        sx = sy = r(0.85, 1.1)
    if "crop" in ops:
        s, ar = r(0.65, 1.0), r(0.9, 1.1).sqrt(); sx, sy = (s * ar).clamp(max=1), (s / ar).clamp(max=1)
        tx, ty = (torch.rand(B, device=dev) * 2 - 1) * (1 - sx), (torch.rand(B, device=dev) * 2 - 1) * (1 - sy)
    if "shift" in ops:
        tx, ty = tx + r(-0.05, 0.05), ty + r(-0.05, 0.05)
    a = r(-1, 1) * math.radians(4) if "rot" in ops else torch.zeros(B, device=dev)
    k = r(-1, 1) * math.tan(math.radians(5)) if "shear" in ops else torch.zeros(B, device=dev)
    theta = torch.zeros(B, 2, 3, device=dev)
    R = torch.stack([torch.stack([a.cos(), -a.sin() * H / W], 1), torch.stack([a.sin() * W / H, a.cos()], 1)], 1)
    Sh = torch.stack([torch.stack([torch.ones(B, device=dev), k * H / W], 1), torch.stack([torch.zeros(B, device=dev), torch.ones(B, device=dev)], 1)], 1)
    S = torch.diag_embed(torch.stack([sx, sy], 1))
    theta[:, :, :2] = S @ R @ Sh; theta[:, 0, 2], theta[:, 1, 2] = tx, ty
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    if "persp" in ops:
        gx, gy = grid[..., 0], grid[..., 1]
        a1, b1, g, h = [r(-0.05, 0.05).view(B, 1, 1) for _ in range(2)] + [r(-0.12, 0.12).view(B, 1, 1) for _ in range(2)]
        den = 1 + g * gx + h * gy
        grid = torch.stack([(gx + a1 * gy) / den, (gy + b1 * gx) / den], -1)
    if "elastic" in ops:
        noise = torch.randn(B, 2, 5, 9, device=dev) * 0.03
        grid = grid + F.interpolate(noise, size=(H, W), mode="bicubic", align_corners=False).permute(0, 2, 3, 1)
    return F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)


def photometric(x, ops):
    if "bright" in ops: x = x * _u(x, 0.75, 1.25)
    if "contrast" in ops:
        m = x.mean((1, 2, 3), keepdim=True); x = (x - m) * _u(x, 0.75, 1.25) + m
    if "sat" in ops:
        g = _gray(x); x = (x - g) * _u(x, 0.7, 1.3) + g
    if "hue" in ops:
        th = _u(x, -0.17, 0.17); c, s = th.cos(), th.sin(); r_, g_, b_ = x[:, 0:1], x[:, 1:2], x[:, 2:3]
        Y = 0.299 * r_ + 0.587 * g_ + 0.114 * b_; I = 0.596 * r_ - 0.274 * g_ - 0.322 * b_; Q = 0.211 * r_ - 0.523 * g_ + 0.312 * b_
        I2, Q2 = I * c - Q * s, I * s + Q * c
        x = torch.cat([Y + 0.956 * I2 + 0.621 * Q2, Y - 0.272 * I2 - 0.647 * Q2, Y - 1.106 * I2 + 1.703 * Q2], 1)
    x = x.clamp(0, 1)
    if "gamma" in ops: x = x.clamp(1e-4, 1) ** _u(x, 0.8, 1.25)
    if "gray" in ops:
        m = _p(x, 0.2); x = m * _gray(x).expand_as(x) + (1 - m) * x
    if "autocontrast" in ops:
        f = x.flatten(1); lo, hi = torch.quantile(f, 0.01, 1).view(-1, 1, 1, 1), torch.quantile(f, 0.99, 1).view(-1, 1, 1, 1)
        m = _p(x, 0.3); x = m * ((x - lo) / (hi - lo + 1e-4)).clamp(0, 1) + (1 - m) * x
    if "posterize" in ops:
        L = (2 ** torch.randint(4, 7, (x.shape[0], 1, 1, 1), device=x.device)).float() - 1
        m = _p(x, 0.2); x = m * (x * L).round() / L + (1 - m) * x
    if "equalize" in ops:
        f = x.flatten(2); N = f.shape[2]
        eq = f.argsort(2).argsort(2).float().div(N - 1).view_as(x)
        m = _p(x, 0.2); x = m * eq + (1 - m) * x
    return x.clamp(0, 1)


def quality(x, ops):
    B, _, H, W = x.shape
    if "blur" in ops:
        m = _p(x, 0.5); x = m * _conv(x, _gauss_k(float(0.3 + 0.9 * torch.rand(1)))) + (1 - m) * x
    if "motion" in ops:
        m = _p(x, 0.3); x = m * _conv(x, _line_k(float(torch.rand(1)) * math.pi, 5 + 4 * float(torch.rand(1)))) + (1 - m) * x
    if "noise" in ops: x = x + torch.randn_like(x) * _u(x, 0.01, 0.05) * _p(x, 0.5)
    if "sp" in ops:
        rnd = torch.rand(B, 1, H, W, device=x.device); amt = _u(x, 0.005, 0.02) * _p(x, 0.3)
        x = torch.where(rnd < amt / 2, torch.zeros_like(x), torch.where(rnd > 1 - amt / 2, torch.ones_like(x), x))
    if "jpeg" in ops:
        arr = (x.clamp(0, 1) * 255).round().byte().permute(0, 2, 3, 1).cpu().numpy(); out = []
        for im in arr:
            if np.random.rand() < 0.5:
                im = cv2.cvtColor(cv2.imdecode(cv2.imencode(".jpg", cv2.cvtColor(im, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, int(np.random.randint(30, 86))])[1], cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
            out.append(im)
        x = torch.from_numpy(np.stack(out)).to(x.device).permute(0, 3, 1, 2).float() / 255
    if "downup" in ops:
        f = float(0.4 + 0.35 * torch.rand(1))
        d = F.interpolate(F.interpolate(x, scale_factor=f, mode="area"), size=(H, W), mode="bilinear", align_corners=False)
        m = _p(x, 0.5); x = m * d + (1 - m) * x
    if "sharpen" in ops:
        m = _p(x, 0.3); x = x + m * _u(x, 0.5, 1.5) * (x - _conv(x, _gauss_k(1.0)))
    if "patch" in ops:
        m = _p(x, 0.5); x = x * (1 + m * _u(x, -0.4, 0.4) * _blob(x, 0.1, 0.35))
    return x.clamp(0, 1)


def weather(x, ops):
    B, _, H, W = x.shape
    if "fog" in ops:
        a = _u(x, 0.1, 0.35) * _p(x, 0.3); x = x * (1 - a) + 0.85 * a
    if "shadow" in ops:
        th = _u(x, 0, 2 * math.pi); X = torch.linspace(-1, 1, W, device=x.device).view(1, 1, 1, W); Y = torch.linspace(-1, 1, H, device=x.device).view(1, 1, H, 1)
        edge = torch.sigmoid(8 * (X * th.cos() + Y * th.sin() - _u(x, -0.4, 0.4)))
        x = x * (1 - _p(x, 0.3) * _u(x, 0.3, 0.6) * edge)
    if "flare" in ops:
        tint = torch.tensor([1.0, 0.9, 0.7], device=x.device).view(1, 3, 1, 1)
        x = x + _p(x, 0.3) * _u(x, 0.3, 0.8) * _blob(x, 0.05, 0.2) * tint
    if "rain" in ops:
        dots = (torch.rand(B, 1, H, W, device=x.device) < 0.002).float()
        x = x + _p(x, 0.3) * 0.5 * _conv(dots.expand(B, 3, H, W).contiguous(), _line_k(math.radians(100 + 10 * float(torch.rand(1))), 9) * 9).clamp(0, 1)
    if "snow" in ops:
        dots = (torch.rand(B, 1, H, W, device=x.device) < 0.003).float()
        x = x + _p(x, 0.3) * 0.8 * _conv(dots.expand(B, 3, H, W).contiguous(), _gauss_k(0.8, 3))
    return x.clamp(0, 1)


def hiding(x, ops):
    fill = torch.rand(x.shape[0], 3, 1, 1, device=x.device)
    if "erase" in ops:
        m = _boxes(x, 1, 0.15, 0.6, 0.1, 0.4) & (_p(x, 0.5) > 0); x = torch.where(m, fill.expand_as(x), x)
    if "cutout" in ops:
        m = _boxes(x, 2, 0.2, 0.3, 0.12, 0.17) & (_p(x, 0.5) > 0); x = torch.where(m, torch.full_like(x, 0.5), x)
    if "griddrop" in ops:
        B, _, H, W = x.shape; unit = int(np.random.randint(32, 57))
        cells = (torch.rand(B, 1, math.ceil(H / unit), math.ceil(W / unit), device=x.device) < 0.3).float()
        m = (F.interpolate(cells, size=(H, W), mode="nearest") > 0) & (_p(x, 0.5) > 0); x = torch.where(m, torch.full_like(x, 0.5), x)
    if "coarse" in ops:
        m = _boxes(x, 10, 0.05, 0.14, 0.03, 0.08) & (_p(x, 0.5) > 0); x = torch.where(m, torch.full_like(x, 0.5), x)
    return x


def mixing(x, y1h, ops):
    """Mixup / CutMix on the whole batch (half of the batches): returns mixed images and soft labels."""
    kinds = [o for o in ("mixup", "cutmix") if o in ops]
    if not kinds or np.random.rand() > 0.5:
        return x, None
    kind, B, perm = kinds[np.random.randint(len(kinds))], x.shape[0], torch.randperm(x.shape[0], device=x.device)
    lam = float(np.random.beta(0.4, 0.4))
    if kind == "mixup":
        return lam * x + (1 - lam) * x[perm], lam * y1h + (1 - lam) * y1h[perm]
    H, W = x.shape[2:]; ch, cw = int(H * math.sqrt(1 - lam)), int(W * math.sqrt(1 - lam))
    cy, cx = np.random.randint(H), np.random.randint(W); y0, y1, x0, x1 = max(cy - ch // 2, 0), min(cy + ch // 2, H), max(cx - cw // 2, 0), min(cx + cw // 2, W)
    x = x.clone(); x[:, :, y0:y1, x0:x1] = x[perm][:, :, y0:y1, x0:x1]
    lam = 1 - (y1 - y0) * (x1 - x0) / (H * W)
    return x, lam * y1h + (1 - lam) * y1h[perm]


def augment(x, y1h, ops):
    """x float (B,3,H,W) in [0,1], y1h one-hot (B,C). Returns the augmented images and soft labels (None when no label mixing happened)."""
    x = hiding(weather(quality(photometric(geometry(x, ops), ops), ops), ops), ops).clamp(0, 1)
    return mixing(x, y1h, ops)


OPS = parse(AUG)
print("augmentasi:", OPS)

# Fine-tuning DINOv3

### Model
Blok 1 sampai 32-K dibekukan, K blok teratas dan norm akhir dilatih, kepala 8 aksi di atasnya. Bobot DINOv3 dibaca dari dataset Kaggle kalau ada (file model.safetensors), kalau tidak diunduh dari Hugging Face.

In [ ]:
MODEL = "vit_huge_plus_patch16_dinov3.lvd1689m"
local = [p for p in Path("/kaggle/input").rglob("*.safetensors") if p.stat().st_size > 2e9]
kw = dict(pretrained_cfg_overlay=dict(file=str(local[0]))) if local else {}
if not local:
    try:
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("HF_TOKEN tidak ada, lanjut tanpa token:", type(e).__name__)
print("bobot DINOv3:", local[0] if local else "unduh dari Hugging Face")
base = timm.create_model(MODEL, pretrained=True, num_classes=0, img_size=IMG_HW, **kw).to(DEVICE).eval()
MEAN = torch.tensor(base.pretrained_cfg["mean"], device=DEVICE).view(1, 3, 1, 1); STD = torch.tensor(base.pretrained_cfg["std"], device=DEVICE).view(1, 3, 1, 1)
EMBED, NPRE, NF = base.embed_dim, base.num_prefix_tokens, 32 - K
AMP = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16


def load_batch(idx):
    arr = [cv2.cvtColor(cv2.resize(cv2.imdecode(np.frombuffer(jpegs[i], np.uint8), cv2.IMREAD_COLOR), (IMG_HW[1], IMG_HW[0]), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB) for i in idx]
    return torch.from_numpy(np.stack(arr))


def to_input(raw):
    return (raw.to(DEVICE).permute(0, 3, 1, 2).float().div(255) - MEAN) / STD


def frozen_forward(x, autocast=True):
    with torch.no_grad(), torch.autocast(DEVICE, dtype=AMP, enabled=autocast):
        t = base.patch_embed(x); t, rope = base._pos_embed(t); t = base.norm_pre(t)
        for b in base.blocks[:NF]:
            t = b(t, rope=rope)
    return t, rope


test_t, _ = frozen_forward(to_input(load_batch(np.arange(0, 2000, 125)[:16])))
FROZEN_AUTOCAST = bool(torch.isfinite(test_t.float()).all())
print("dtype AMP", AMP, "| blok beku dalam", "AMP" if FROZEN_AUTOCAST else "fp32 (AMP menghasilkan nilai tak hingga)")
if FROZEN_AUTOCAST:
    for b in base.blocks[:NF]:
        b.to(AMP)
for p in base.parameters():
    p.requires_grad_(False)
train_blocks = nn.ModuleList(base.blocks[NF:])
for p in list(train_blocks.parameters()) + list(base.norm.parameters()):
    p.requires_grad_(True)
blocks_params = list(train_blocks.parameters()) + list(base.norm.parameters())
print(f"parameter dilatih {sum(p.numel() for p in blocks_params) / 1e6:.0f}M | memori {torch.cuda.memory_allocated() / 1e9:.1f} GB")


def make_head():
    return nn.Sequential(nn.BatchNorm1d(2 * EMBED, affine=False), nn.Dropout(0.2), nn.Linear(2 * EMBED, len(ACTIONS))).to(DEVICE)


def forward(x, head, training):
    t, rope = frozen_forward(x, FROZEN_AUTOCAST)
    with torch.autocast(DEVICE, dtype=AMP):
        for b in train_blocks:
            t = checkpoint(b, t, rope=rope, use_reentrant=False) if training else b(t, rope=rope)
        t = base.norm(t)
        f = torch.cat([t[:, 0], t[:, NPRE:].mean(1)], 1)
    return head(f.float()), f.float()

### Fungsi training
Fungsi yang dipakai: urutan batch, langkah training dengan augmentasi, penyimpanan weights, dan rata-rata weights (EMA) selama training.


In [ ]:
y_np = meta["y"].values


def make_order():
    wr, act = tr_idx[y_np[tr_idx] == WRITE], tr_idx[y_np[tr_idx] != WRITE]
    return np.random.permutation(np.r_[act, np.random.choice(wr, len(wr) // 3, replace=False)])


def batches(order, bs):
    chunks = [np.sort(order[s:s + bs]) for s in range(0, len(order), bs)]
    with ThreadPoolExecutor(3) as pool:
        futs = [pool.submit(load_batch, c) for c in chunks[:4]]
        for k, c in enumerate(chunks):
            raw = futs.pop(0).result()
            if k + 4 < len(chunks):
                futs.append(pool.submit(load_batch, chunks[k + 4]))
            yield c, raw
y_gpu = torch.tensor(y_np, device=DEVICE)


all_params = lambda head: blocks_params + list(head.parameters())


def make_opt(head, lr_scale=1.0):
    return torch.optim.AdamW([{"params": blocks_params, "lr": LR_BLOCKS * lr_scale}, {"params": list(head.parameters()), "lr": LR_HEAD * lr_scale}], weight_decay=WD)


def augmented_input(raw):
    x = raw.to(DEVICE).permute(0, 3, 1, 2).float().div(255)
    x, _ = augment(x, None, OPS)
    return (x - MEAN) / STD


def train_step(head, opt, scaler, c, raw):
    loss = F.cross_entropy(forward(augmented_input(raw), head, True)[0], y_gpu[torch.from_numpy(c).to(DEVICE)], weight=w, label_smoothing=0.1)
    opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(all_params(head), 1.0); scaler.step(opt); scaler.update()
    return loss


def save_weights(path):
    torch.save({"K": K, "blocks": {NF + i: {k: v.half().cpu() for k, v in b.state_dict().items()} for i, b in enumerate(train_blocks)},
                "norm": {k: v.half().cpu() for k, v in base.norm.state_dict().items()}}, path)


def ema_update(ema, params, step, total_steps):
    # decay so that the average covers about EMA_FRAC of the steps; early steps use a smaller decay so the start weights wash out
    d = min(1 - 1 / max(1.0, EMA_FRAC * total_steps), (1 + step) / (10 + step))
    with torch.no_grad():
        torch._foreach_mul_(ema, d); torch._foreach_add_(ema, [p.detach().float() for p in params], alpha=1 - d)


### Training
Ukur kecepatan GPU, latih dengan augmentasi di seluruh frame train sambil merata-rata weights (EMA), lalu simpan weights tanpa rata-rata (final) dan weights rata-rata (dipakai apa adanya untuk inferensi).


In [ ]:
TAG = f"k{K}_{FPS}fps_aug_full"
tr_idx = np.where(meta["y"].values >= 0)[0]
print("frame training", len(tr_idx), "| per kelas", np.bincount(meta["y"].values[tr_idx], minlength=8))
np.random.seed(SEED); torch.manual_seed(SEED)
order0 = make_order()
cnt = np.bincount(y_np[order0], minlength=8)
w = torch.tensor((cnt.sum() / (8 * cnt)) ** 0.5, dtype=torch.float32, device=DEVICE)

head, scaler = make_head(), torch.amp.GradScaler(enabled=(AMP == torch.float16))
init_blocks = [{k: v.detach().cpu().clone() for k, v in b.state_dict().items()} for b in train_blocks]
init_norm = {k: v.detach().cpu().clone() for k, v in base.norm.state_dict().items()}
opt = make_opt(head, 0.0); base.eval(); head.train(); stamps = []
for k, (c, raw) in enumerate(batches(order0[:BS * 16], BS)):
    train_step(head, opt, scaler, c, raw); torch.cuda.synchronize(); stamps.append(time.time())
sec_per_step = (stamps[-1] - stamps[3]) / (len(stamps) - 4)
planned = EPOCHS * math.ceil(len(order0) / BS)
possible = int((TIME_BUDGET_H * 3600 - (time.time() - T0) - 1500) / sec_per_step)
total_steps = max(50, min(planned, possible))
print(f"{sec_per_step:.2f} detik/step ({BS / sec_per_step:.1f} frame/detik) | step rencana {planned} ({planned * sec_per_step / 3600:.1f} jam) | step yang muat {possible} | dipakai {total_steps} ({total_steps * sec_per_step / 3600:.1f} jam)")
del opt, head
for p in blocks_params:
    p.grad = None
for b, s in zip(train_blocks, init_blocks):      # the 16 speed-test steps ran at lr 0, but start from the exact DINOv3 weights anyway
    b.load_state_dict(s)
base.norm.load_state_dict(init_norm); del init_blocks, init_norm
torch.cuda.empty_cache()

head, scaler = make_head(), torch.amp.GradScaler(enabled=(AMP == torch.float16))
opt = make_opt(head); warm = max(1, total_steps // 20)
lr_at = lambda s: min(1, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1, s / total_steps)))
ema_params = all_params(head); ema = [p.detach().clone().float() for p in ema_params]
step, t_train, losses, done = 0, time.time(), [], False
for ep in range(EPOCHS):
    head.train(); order = order0 if ep == 0 else make_order()
    for c, raw in batches(order, BS):
        for g, base_lr in zip(opt.param_groups, (LR_BLOCKS, LR_HEAD)):
            g["lr"] = base_lr * lr_at(step)
        losses.append(float(train_step(head, opt, scaler, c, raw))); step += 1
        ema_update(ema, ema_params, step, total_steps)
        if step % 100 == 0:
            print(f"step {step}/{total_steps} | loss {np.mean(losses[-100:]):.3f} | {(time.time() - t_train) / 60:.0f} menit | sisa {(total_steps - step) * (time.time() - t_train) / step / 60:.0f} menit", flush=True)
        if step % 500 == 0:
            save_weights(OUT / f"{TAG}_final_weights.pt")
        if step >= total_steps:
            done = True; break
    if done:
        break
print("selesai", step, "step |", f"{(time.time() - t_train) / 3600:.2f} jam | loss akhir {np.mean(losses[-100:]):.3f}")

save_weights(OUT / f"{TAG}_final_weights.pt"); torch.save(head.state_dict(), OUT / f"{TAG}_final_head.pt")
with torch.no_grad():                       # swap in the averaged weights (BatchNorm statistics of the head stay as trained)
    for p, e in zip(ema_params, ema):
        p.copy_(e.to(p.dtype))
save_weights(OUT / f"{TAG}_weights.pt"); torch.save(head.state_dict(), OUT / f"{TAG}_head.pt")
summary = dict(tag=TAG, K=K, fps=FPS, epochs=EPOCHS, holdout_fold=None, aug=AUG, ema_frac=EMA_FRAC, steps=step, total_steps_planned=planned, sec_per_step=round(sec_per_step, 3),
               amp=str(AMP), train_hours=round((time.time() - t_train) / 3600, 2), final_loss=float(np.mean(losses[-100:])), total_hours=round((time.time() - T0) / 3600, 2))
(OUT / f"{TAG}_summary.json").write_text(json.dumps(summary, indent=1))
print(summary)


# Output
File yang disimpan di /kaggle/working dan perlu diunduh:
- k12_15fps_aug_full_weights.pt dan k12_15fps_aug_full_head.pt: weights rata-rata (EMA) dari K blok teratas, norm, dan kepala 8 aksi. Ini model yang dipakai untuk inferensi.
- k12_15fps_aug_full_final_weights.pt dan k12_15fps_aug_full_final_head.pt: weights tanpa rata-rata (cadangan, untuk membandingkan).
- k12_15fps_aug_full_summary.json: konfigurasi dan loss akhir.


In [ ]:
for p in sorted(OUT.glob("k*")):
    print(f"{p.name:50s} {p.stat().st_size / 1e6:8.1f} MB")